# ML-1M: Sparse Walker + residual SWG — train from scratch

Random initialization: no pretrained Walker, no pretrained navigation query. The original K=8 Walker trains on all autoregressive positions; the final prediction position of each sampled window is refined by a 4-hop / beam-16 residual SWG read over a flattened HNSW graph. HNSW is rebuilt every epoch as concept geometry changes.

The runner first asserts that SWG-off evaluation exactly matches the canonical evaluator, then prints live training/evaluation diagnostics.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, sys, subprocess, shutil, runpy
REPO='/content/Sparsewalker'
BRANCH='agent/walker-hybrid-swg-scratch'
if os.path.exists(REPO): shutil.rmtree(REPO)
subprocess.run(['git','clone','-q','-b',BRANCH,'https://github.com/hanialshater/Sparsewalker-.git',REPO], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','faiss-cpu'], check=True)
SRC=f'{REPO}/src'
EXPS=f'{REPO}/experiments'
sys.path.insert(0,SRC)
sys.path.insert(0,EXPS)
os.chdir(REPO)

sys.argv=[
    'run_ml1m_hybrid_swg_scratch.py',
    '--seed','42',
    '--max-epochs','60',
    '--eval-every','5',
    '--patience','20',
    '--batch-size','128',
    '--eval-batch-size','512',
    '--hnsw-m','8',
    '--ef-construction','160',
    '--hops','4',
    '--beam','16',
]
print('INPROCESS SCRATCH HYBRID START', flush=True)
runpy.run_path(f'{EXPS}/run_ml1m_hybrid_swg_scratch.py', run_name='__main__')
print('INPROCESS SCRATCH HYBRID END', flush=True)

### What to send back

First send `EVALUATOR_ASSERT` and the first `EVAL` line. If training continues normally, the decisive outputs are the best later `EVAL`, `HISTORY_LENGTH`, and `FINAL_RESULT`.